# Workshop 3 — Build a Neural Network That Reads Handwriting

You will build **one real Python file** while the slides explain each idea exactly when you need it.

Keep this loop in mind: **Forward Pass → Loss → Backpropagation → Update Weights → Repeat**.

## 0. Set up your workbench
Run this once. Use the standard **CPU** runtime—no GPU is needed. The cell downloads the workshop, creates your editable file, checks PyTorch, and preloads MNIST.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

import torch
import torchvision

REPO_URL = "https://github.com/daryl-888/Workshop3.git"
REPO_DIR = Path("/content/Workshop3")
WORK_FILE = Path("/content/mnist_network.py")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

if not WORK_FILE.exists():
    shutil.copy(REPO_DIR / "starter/mnist_network.py", WORK_FILE)
    work_file_status = "Created your starter file"
else:
    work_file_status = "Kept your existing file (setup is safe to rerun)"

for train in (True, False):
    torchvision.datasets.MNIST("/content/data", train=train, download=True)

print(f"Python {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()} (we are intentionally using CPU)")
print(f"{work_file_status}: {WORK_FILE}")

## 1. Open your Python file
In Colab's left sidebar, click the **folder** icon, then double-click `mnist_network.py`. Keep this notebook open for checkpoints while you edit the Python file beside it.

## Checkpoint 1 — your Linear layer
Save your Python file, then run this. A green check means three images made it through `784 → 128`.

In [ ]:
# CHECKPOINT 1: run the student's file in a fresh Python process.
checkpoint = '''
import torch
from mnist_network import Linear
output = Linear(784, 128).forward(torch.zeros(3, 784))
assert output.shape == (3, 128), output.shape
print("✅ Linear checkpoint passed: (3, 784) → (3, 128)")
'''
subprocess.run([sys.executable, "-c", checkpoint], cwd="/content", check=True)

## Checkpoint 2 — your complete network
Save again, then confirm four images become ten digit scores each.

In [ ]:
# CHECKPOINT 2: import the saved file from scratch.
checkpoint = '''
import torch
from mnist_network import NeuralNetwork
model = NeuralNetwork(784, 128, 10)
output = model.forward(torch.zeros(4, 784))
assert output.shape == (4, 10), output.shape
assert len(model.parameters()) == 6
print("✅ Network checkpoint passed: (4, 784) → (4, 10)")
'''
subprocess.run([sys.executable, "-c", checkpoint], cwd="/content", check=True)

## Checkpoint 3 — one learning step
This tiny test makes a guess, measures the loss, walks backward, changes the weights, and checks that the next loss is smaller.

In [ ]:
# CHECKPOINT 3: prove the manual learning loop changes the model.
checkpoint = '''
import torch
from mnist_network import NeuralNetwork
torch.manual_seed(0)
model = NeuralNetwork(2, 4, 2)
x = torch.tensor([[2.0, 0.0], [0.0, 2.0]])
target = torch.tensor([0, 1])
before = model.forward(x)
loss_before = torch.nn.functional.cross_entropy(before, target)
loss_before.backward()
with torch.no_grad():
    for parameter in model.parameters():
        parameter -= 0.1 * parameter.grad
for parameter in model.parameters():
    parameter.grad.zero_()
loss_after = torch.nn.functional.cross_entropy(model.forward(x), target)
assert loss_after < loss_before
print(f"✅ Learning checkpoint passed: {loss_before.item():.4f} → {loss_after.item():.4f}")
'''
subprocess.run([sys.executable, "-c", checkpoint], cwd="/content", check=True)

## Recovery lane
Only use this if the facilitator tells you to. It replaces your current file with a known-good checkpoint so you can rejoin the room.

In [ ]:
def recover(stage):
    names = {
        1: "01-linear.py",
        2: "02-network.py",
        3: "03-learning-step.py",
        4: "04-complete.py",
    }
    if stage not in names:
        raise ValueError("stage must be 1, 2, 3, or 4")
    source = REPO_DIR / "facilitator/checkpoints" / names[stage]
    shutil.copy(source, WORK_FILE)
    print(f"Recovered stage {stage}: {names[stage]}")

### Recovery buttons
Run only the stage your facilitator names. **Recovery replaces your current file**, so download it first if you want to keep your attempt.

In [ ]:
# Recover after the Linear checkpoint.
recover(1)

In [ ]:
# Recover after the network checkpoint.
recover(2)

In [ ]:
# Recover after the learning-step checkpoint.
recover(3)

In [ ]:
# Recover the complete workshop solution.
recover(4)

## Final run — watch it learn
Save `mnist_network.py`, run it for 10 epochs, then change `epochs = 10` to `epochs = 60` and run again. Loss should fall while test accuracy rises.

In [ ]:
# Final run: execute exactly the file the student built.
subprocess.run([sys.executable, str(WORK_FILE)], cwd="/content", check=True)

## Keep what you built
Download your finished Python file before the temporary Colab runtime disappears.

In [ ]:
from google.colab import files
files.download(str(WORK_FILE))